# Assingment 3
Matthew gilvear u5597048

## Initial usecase of Python
I wanted to practice the intuition behind both Nim and Build Nim. I implemented a function that would randomly build starting positions given the pair $(n,s)$ and a function that would allow you to manually build them as well. Using this and the funtion play() which allows me to play through a game of Nim, I was able to get an initial understanding and feel for how the game worked.

In [10]:
import random as rd
import numpy as np

#Global player variable to alternate
player  = 0


def build_random(n,s):
    piles = np.zeros(n,dtype=int)
    for i in range(s):
        piles[rd.randint(0,n-1)] += 1
    return piles

def build(n,s):
    global player
    #Initiate empty piles
    piles = np.zeros(n,dtype=int)

    while sum(piles)<s:

        print(f"Player {player}: {piles}")
        p = int(input("Choose a pile: "))

        if p < 0 or p >= len(piles):
            print("Invalid pile. Try again.")
            continue

        piles[p] += 1
        player = (player+1)%2
    
    return piles    

In [11]:
def play(piles):
    global player

    while sum(piles) > 0:
        #Which Pile to choose
        print(f"Player {player}: {piles}")
        p = int(input("Choose a pile: "))

        #Checks Valid pile
        if p < 0 or p >= len(piles) or piles[p] == 0:
            print("Invalid pile. Try again.")
            continue

        #Takes how many to remove and if valid
        m = int(input("How many to remove: "))
        if m <= 0 or m > piles[p]:
            print("Invalid number. Try again.")
            continue
        
        player = (player+1)%2
        piles[p] -= m       

            
#play(build_random(5,21))
#play(build(5,21))

## Defining winning starting positions
the usecase of Nim SUm is clear given a position it calculates the Nim Sum as described in the Entry Phase of my rubric. The second function return winner uses the logic I developed in the Attack Phase of my answer to return the winner given a starting position. Either Player 0 or Player 1.

In [12]:
import math

def nim_sum(piles):
    width = math.floor(math.log(max(piles),2)) + 1
    grid = np.zeros((len(piles),width))
    for i in range(len(piles)):
        n = piles[i]
        power = 1
        while n>0:
            binary_digit = n%(2**(power))
            if binary_digit:
                grid[i][-power] = 1
            n -= binary_digit*(2**(power-1))
            power+=1

    row_vector = grid.sum(axis=0)
    row_vector = row_vector%2
    row_vector = row_vector * 2**np.arange(len(row_vector)-1,-1,-1)
    
    return sum(row_vector)

def return_winner(piles):
    m = (np.array(piles)>1).sum()
    if m == 0:
        return sum(piles)%2
    else:
        n_s = nim_sum(piles)
        if n_s == 0:
            return 1
        else:
            return 0

## Building initial starting positions
The function outlined below was my first attempt to build all starting positions. It does this very much by a brute force approach using the package iterools.
It first generates the set of tuples \[\{(a_0, \ldots a_{n-1}):0 \leq a_i \leq s \}\]
Then it refines it to the set of tuples \[\{(a_0, \ldots a_{n-1}): \sum a_i = s\} \subset \{(a_0, \ldots a_{n-1}):0 \leq a_i \leq s \}\]
Then it refines it again to the set of tuples \[\{(a_0, \ldots a_{n-1}): \sum a_i = s, a_i \leq a_j \iff i \leq j\}\] this is the set up tuples that we want.

In [13]:
from itertools import product
def starting_positions(n,s):
    all_tuples = list(product(range(s+1), repeat=n))
    correct_sum = [t for t in all_tuples if sum(t) == s]
    correct_order = set(tuple(sorted(t)) for t in correct_sum)
    return sorted(list(correct_order))

#starting_positions(3,5)

The below function has exactly the same output but was generated by AI to make the process much much faster

In [14]:
def find_sum_tuples_optimized(n: int, s: int) -> list[tuple]:
    """
    Finds all unique tuples of length n that sum up to s (Partitions of S into N parts).

    Args:
        n: The required length of the tuple.
        s: The target sum.

    Returns:
        A sorted list of unique tuples satisfying the criteria.
    """
    if n <= 0 or s < 0:
        return []

    results = set()

    def backtrack(index, current_sum, current_tuple):
        """
        Recursive helper function to build the combinations using strict partitioning rules.
        """
        # 1. Base Case: We have filled all N positions
        if index == n:
            if current_sum == s:
                results.add(tuple(current_tuple))
            return

        # Optimization Check (Pruning): If the sum already exceeds S, stop immediately
        if current_sum > s:
            return

        # 2. Determine the range for the current element (a_index)
        
        # The minimum value we can place at this index is determined by:
        # a) It must be at least 0 (or some starting point, depending on problem constraints).
        # b) To maintain non-decreasing order, it must be >= the previous element.
        start_val = current_tuple[-1] if index > 0 else 0

        # The maximum value we can place is determined by:
        # a) The total remaining sum (s - current_sum).
        # b) We must ensure that even if we pick the smallest possible number for all remaining slots, the sum will still reach S.
        
        # Calculate the theoretical upper bound for this slot.
        # Max value = Remaining Target Sum - Minimum Required Sum for the rest of the slots.
        remaining_slots = n - index
        min_needed_for_rest = 0 # Since we allow 0s, the minimum is just 0. (If we required positive integers, this would be different.)

        # A simpler and more robust bound check: The current value cannot exceed the total remaining sum.
        # If we choose 'val', the remaining slots (n - index - 1) must be able to hold the remaining target sum (s - current_sum - val).
        
        max_val = s - current_sum # Simplest upper bound: don't overshoot S

        for val in range(start_val, max_val + 1):
            # Recursive Call: Place the value and move to the next position
            backtrack(
                index + 1,
                current_sum + val,
                current_tuple + [val]
            )

    # Start the backtracking process
    backtrack(0, 0, [])
    
    return sorted(list(results))


Here is an illustration of how much faster.
We still see exactly the same result as well!

In [15]:
#Drag racing
import time 
import time

start = time.perf_counter()
result1 = starting_positions(5, 25)
elapsed = time.perf_counter() - start
print(f"My function took {elapsed} s")

start = time.perf_counter()
result2 = find_sum_tuples_optimized(5, 25)
elapsed = time.perf_counter() - start
print(f"The AI function took {elapsed} s")


print(result1)
print(result2)

My function took 5.712417299975641 s
The AI function took 0.0049912999966181815 s
[(0, 0, 0, 0, 25), (0, 0, 0, 1, 24), (0, 0, 0, 2, 23), (0, 0, 0, 3, 22), (0, 0, 0, 4, 21), (0, 0, 0, 5, 20), (0, 0, 0, 6, 19), (0, 0, 0, 7, 18), (0, 0, 0, 8, 17), (0, 0, 0, 9, 16), (0, 0, 0, 10, 15), (0, 0, 0, 11, 14), (0, 0, 0, 12, 13), (0, 0, 1, 1, 23), (0, 0, 1, 2, 22), (0, 0, 1, 3, 21), (0, 0, 1, 4, 20), (0, 0, 1, 5, 19), (0, 0, 1, 6, 18), (0, 0, 1, 7, 17), (0, 0, 1, 8, 16), (0, 0, 1, 9, 15), (0, 0, 1, 10, 14), (0, 0, 1, 11, 13), (0, 0, 1, 12, 12), (0, 0, 2, 2, 21), (0, 0, 2, 3, 20), (0, 0, 2, 4, 19), (0, 0, 2, 5, 18), (0, 0, 2, 6, 17), (0, 0, 2, 7, 16), (0, 0, 2, 8, 15), (0, 0, 2, 9, 14), (0, 0, 2, 10, 13), (0, 0, 2, 11, 12), (0, 0, 3, 3, 19), (0, 0, 3, 4, 18), (0, 0, 3, 5, 17), (0, 0, 3, 6, 16), (0, 0, 3, 7, 15), (0, 0, 3, 8, 14), (0, 0, 3, 9, 13), (0, 0, 3, 10, 12), (0, 0, 3, 11, 11), (0, 0, 4, 4, 17), (0, 0, 4, 5, 16), (0, 0, 4, 6, 15), (0, 0, 4, 7, 14), (0, 0, 4, 8, 13), (0, 0, 4, 9, 12), (0, 0, 

#Brief exploration into the distribution of winning positions
These tables illustrate all starting positions given the pairs (n,s) and the starting psotions that are winning for player 1.  It was really difficult to find a good way to print these and I don't think the results were particualrly useful in the end so they weren't included in the latex rubric, but ruling it out was an important part of the problem solving process. 

In [16]:
import pandas as pd
n = np.arange(1,8)
s = np.arange(1,16)
table_all = pd.DataFrame([[find_sum_tuples_optimized(a, b) for b in s] for a in n],
                        index=pd.Index(n,name="n"),
                        columns=pd.Index(s,name="s"))
table_winning = pd.DataFrame([[[t for t in find_sum_tuples_optimized(a, b) if   return_winner(t) == 1] for b in s] for a in n],
                     index=pd.Index(n,name="n"),
                     columns=pd.Index(s,name="s"))
print(table_winning.loc[3]) 
print(table_all.loc[3])
print(table_winning.loc[5])
print(table_winning.loc[7])

s
1                                           [(0, 0, 1)]
2                                                    []
3                                           [(1, 1, 1)]
4                                           [(0, 2, 2)]
5                                                    []
6                                [(0, 3, 3), (1, 2, 3)]
7                                                    []
8                                           [(0, 4, 4)]
9                                                    []
10                               [(0, 5, 5), (1, 4, 5)]
11                                                   []
12                               [(0, 6, 6), (2, 4, 6)]
13                                                   []
14    [(0, 7, 7), (1, 6, 7), (2, 5, 7), (3, 4, 7), (...
15                                                   []
Name: 3, dtype: object
s
1                                           [(0, 0, 1)]
2                                [(0, 0, 2), (0, 1, 1)]
3                    

## Implemenataion of Child Theory

This is done in 4 functions. Function 1 generates the inital set of good nodes. Function 2 returns a list of every parent of a set of good nodes. Function 3 returns a list of every parent such that every child of a parent in that list is a good child. Function 4 ties all the functions together to determine if Player 1 can win the game. The results are then calculated in a Pandas dataframe.

In [17]:
#print(find_sum_tuples_optimized(3, 6))
def initial_good_nodes(n,s):
    return[g for g in find_sum_tuples_optimized(n,s) if return_winner(g) == 1]

#print(initial_good_nodes(3,6))

def player_1_good_nodes(positions):
    result = []
    n = len(positions[0])
    for position in positions:
        for i in range(n):
            if position[i] - 1 >= 0:
                new_position = list(position)
                new_position[i] -= 1
                new_position.sort()
                if tuple(new_position) not in result:
                    result.append(tuple(new_position))
    return result

def player_0_good_nodes(positions):
    result = []
    n = len(positions[0])
    s = sum(positions[0])
    possible = find_sum_tuples_optimized(n,s-1)
    for position in possible:
        good = True
        for i in range(n):
            new_position = list(position)
            new_position[i] += 1
            new_position.sort()
            if tuple(new_position) not in positions:
                good = False
                break
        if good and position not in result:
            result.append(position)
    return result

'''
print(initial_good_nodes(3,6))
print("result1",player_1_good_nodes(initial_good_nodes(3,6)))
print("result2",player_0_good_nodes(player_1_good_nodes(initial_good_nodes(3,6))))
print("result3",player_1_good_nodes(player_0_good_nodes(player_1_good_nodes(initial_good_nodes(3,6)))))
print("result4",player_0_good_nodes(player_1_good_nodes(player_0_good_nodes(player_1_good_nodes(initial_good_nodes(3,6))))))

'''

#takes n and s even
def game_winner(n,s):
    good_nodes = initial_good_nodes(n,s)
    #print(good_nodes)
    while good_nodes and not good_nodes == [(0,)*n]:
        good_nodes = player_1_good_nodes(good_nodes)
        #print(good_nodes)
        good_nodes = player_0_good_nodes(good_nodes)
        #print(good_nodes)
    if good_nodes == [(0,)*n]:
        return 1
    else:
        return 0

n = np.arange(1,14)
s = np.arange(2,35,2)

final_table = pd.DataFrame([[game_winner(a, b) for b in s] for a in n],
                        index=pd.Index(n,name="n"),
                        columns=pd.Index(s,name="s"))

print(final_table)



s   2   4   6   8   10  12  14  16  18  20  22  24  26  28  30  32  34
n                                                                     
1    0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
2    0   1   1   1   1   1   1   1   1   1   1   1   1   1   1   1   1
3    0   0   1   0   0   0   1   0   0   0   0   0   0   0   0   0   0
4    0   0   1   1   1   1   1   1   1   1   1   1   1   1   1   1   1
5    0   0   1   1   0   0   0   0   0   0   0   0   0   0   0   0   0
6    0   0   1   1   1   1   1   1   1   1   1   1   1   1   1   1   1
7    0   0   1   1   1   0   0   0   0   0   0   0   0   0   0   0   0
8    0   0   1   1   1   1   1   1   1   1   1   1   1   1   1   1   1
9    0   0   1   1   1   1   0   0   0   0   0   0   0   0   0   0   0
10   0   0   1   1   1   1   1   1   1   1   1   1   1   1   1   1   1
11   0   0   1   1   1   1   1   0   0   0   0   0   0   0   0   0   0
12   0   0   1   1   1   1   1   1   1   1   1   1   1   1   1   1   1
13   0

This bit of code just adds in what I developed in the specialisation early on in the attack phase.  

In [18]:
s = np.arange(1,18)

full_table = pd.DataFrame([[1 if b==1 else 0 if b%2 == 1 else game_winner(a, b) for b in s] for a in n],
                        index=pd.Index(n,name="n"),
                        columns=pd.Index(s,name="s"))

print(full_table)

s   1   2   3   4   5   6   7   8   9   10  11  12  13  14  15  16  17
n                                                                     
1    1   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
2    1   0   0   1   0   1   0   1   0   1   0   1   0   1   0   1   0
3    1   0   0   0   0   1   0   0   0   0   0   0   0   1   0   0   0
4    1   0   0   0   0   1   0   1   0   1   0   1   0   1   0   1   0
5    1   0   0   0   0   1   0   1   0   0   0   0   0   0   0   0   0
6    1   0   0   0   0   1   0   1   0   1   0   1   0   1   0   1   0
7    1   0   0   0   0   1   0   1   0   1   0   0   0   0   0   0   0
8    1   0   0   0   0   1   0   1   0   1   0   1   0   1   0   1   0
9    1   0   0   0   0   1   0   1   0   1   0   1   0   0   0   0   0
10   1   0   0   0   0   1   0   1   0   1   0   1   0   1   0   1   0
11   1   0   0   0   0   1   0   1   0   1   0   1   0   1   0   0   0
12   1   0   0   0   0   1   0   1   0   1   0   1   0   1   0   1   0
13   1